## Environment Setup

In [8]:
%%capture
!apt-get install -y ffmpeg
!pip install -q google-genai

import os
import pandas as pd

## Audio File Selection and Google Drive Setup

In [10]:
from google.colab import drive
import os

# 1. Mount Google Drive to pull files directly
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# 2. Define the exact path to your source folder containing your m4a files
# Feel free to edit this path string as your folder structure evolves
SOURCE_FOLDER_PATH = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files"

# 3. Enter the exact audio file name you want to process
audio_filename = input("Enter the specific .m4a file name (e.g., speech_recording.m4a): ").strip()
drive_file_path = os.path.join(SOURCE_FOLDER_PATH, audio_filename)

# 4. Verify file existence and map variable names to keep downstream cells compatible
if os.path.exists(drive_file_path):
    # video_id acts as the base name so your saving folder architecture works flawlessly
    video_id = os.path.splitext(audio_filename)[0]
    print(f"\n✨ File successfully located at: {drive_file_path}")
    print(f"✨ Subfolder database key created: {video_id}")
else:
    print(f"\n❌ Error: '{audio_filename}' was not found in '{SOURCE_FOLDER_PATH}'. Check spelling or paths.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive successfully mounted.
Enter the specific .m4a file name (e.g., speech_recording.m4a): MarauliKhurad3.m4a

✨ File successfully located at: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/MarauliKhurad3.m4a
✨ Subfolder database key created: MarauliKhurad3


## Gemini API Initialization and Model Selection

In [11]:
import google.colab.widgets as widgets
from ipywidgets import Dropdown
from google import genai
from google.colab import userdata

# 1. Initialize the modern unified Gemini developer API client
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Run a rapid validation ping to ensure everything is authenticated correctly
try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Connection successful.'
    )
    print("Gemini API connection verified.")
except Exception as e:
    print(f"Gemini API verification failed: {e}. Confirm your Colab Secret Keys.")

# 3. Set your active production model tier list
gemini_models = [
    'gemini-3.5-flash',
    'gemini-2.0-flash',
    'gemini-1.5-pro',
    'gemini-2.5-flash',
    'gemini-3.1-flash-lite',
    'gemini-3-flash'
]

# 4. Generate user interface selection node
model_selector = Dropdown(
    options=gemini_models,
    value='gemini-3.5-flash',
    description='Select Gemini Model:'
)
display(model_selector)

# 5. Establish global state updates for cell memory tracking
MODEL_NAME = model_selector.value

def on_model_change(change):
    global MODEL_NAME
    MODEL_NAME = change.new
    print(f"Selected model updated to: {MODEL_NAME}")

model_selector.observe(on_model_change, names='value')
print("\nGemini client ready and listening for model selection updates.")

Gemini API connection verified.


Dropdown(description='Select Gemini Model:', options=('gemini-3.5-flash', 'gemini-2.0-flash', 'gemini-1.5-pro'…


Gemini client ready and listening for model selection updates.
Selected model updated to: gemini-2.5-flash
Selected model updated to: gemini-3.1-flash-lite


## Audio Transcription and Interactive Chat

In [15]:
import time

# 1. Direct path routing to your specified Google Drive audio file
file_path = drive_file_path

print(f"Uploading {os.path.basename(file_path)} to Gemini...")
audio_file = client.files.upload(file=file_path)

# Wait briefly for the file API to finish processing the state
print("Waiting for file processing...")
time.sleep(3)

print("\n--- Generating Transcript ---")
# 2. UPDATED: Strict prompt to fetch only the raw transcript with no AI analysis
transcription_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[
        audio_file,
        "Your sole task is to generate a verbatim, word-for-word text transcript of the provided audio file in the exact way the audio same script. Do not include any AI-generated summaries, structural overviews, participant profiles, metrics sections, or extra commentary. Output only the direct conversational dialogue exactly as it is spoken by the participants, preserving their native code-switching and dialect transitions entirely."
    ]
)
print(transcription_response.text)

print("\n--- Starting Conversation Mode ---")
# 3. Create an ongoing chat session with the audio context
chat = client.chats.create(model=MODEL_NAME)

# FIXED: Changed 'contents=' to 'message=' to align with the new SDK
chat.send_message(message=[audio_file, "Analyze this audio file. I will now ask you questions about its contents."])

print("Chat initialized! You can now ask questions about the audio below. (Type 'exit' to stop)")

# 4. Interactive loop to converse with the audio
while True:
    user_input = input("\nYour Question: ")
    if user_input.lower() == 'exit':
        print("Ending chat session.")
        break

    if user_input.strip() == '':
        continue

    response = chat.send_message(user_input)
    print(f"\nGemini: {response.text}")

Uploading MarauliKhurad3.m4a to Gemini...
Waiting for file processing...

--- Generating Transcript ---
ਆ ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ। ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ। ਆਪਣਾ ਨਾਮ ਜੀ? ਦਲੀਪ ਸਿੰਘ ਠੀਕ ਹੈ ਜੀ। ਤੇ ਆਪਣਾ ਪਿੰਡ ਜੀ? ਮਰਾਊਲੀ ਖੁਰਦ। ਠੀਕ ਹੈ ਜੀ ਤੇ ਆਪਾਂ ਸਰ ਕਿੰਨੀ ਕੁ ਜਮੀਨ ਦੀ ਖੇਤੀ ਕਰਦੇ ਹੋ ਜੀ? ਖੇਤੀ ਕਰਦੇ 10-12 ਕਿੱਲੇ ਜੀ। 10-12 ਕਿੱਲੇ ਜੀ। ਤੇ 10-12 ਕਿੱਲੇ 'ਚ ਕਿਹੜੀਆਂ ਕਿਹੜੀਆਂ ਫਸਲਾਂ ਬੀਜਦੇ ਜੀ ਆਪਾਂ? ਬਾਹੀ, ਝੋਨਾ, ਹਾਂਜੀ। ਜਿਹੜੀ ਮਤਲਬ ਕਣਕ ਝੋਨਾ ਔਰ ਕਣਕ ਹਨਾ ਜੀ। ਹਾਂਜੀ। ਬਾਕੀ ਅਰਾਇਆ ਫੱਟੇ ਬਾਖਣ ਨੂੰ ਅੱਛਾ ਜੀ। ਰਾਯਾ ਰੋਯਾ ਸਰੋਂ ਵਗੈਰਾ ਬੀਜਦੇ ਆ ਜੀ। ਹਾਂ ਤੇ ਬੀਜਦੇ ਫਿਰ ਵੀ ਠੀਕ ਆ। ਪਸ਼ੂ ਰੱਖੇ ਆ ਜੀ ਆਪਾਂ? ਹਾਂ ਪਸ਼ੂ ਰੱਖੇ ਆ। ਠੀਕ ਹੈ ਜੀ। ਤੇ ਸਰ ਮੈਂ ਪੁੱਛਣਾ ਚਾਹੁੰਦਾ ਜੀ ਕਣਕ ਵਿੱਚ ਕਿਹੜੀਆਂ-ਕਿਹੜੀਆਂ ਜਿਹੜੀਆਂ ਇੱਦਾਂ ਦੀਆਂ ਬਿਮਾਰੀਆਂ ਐ, ਕਿਹੜੀਆਂ-ਕਿਹੜੀਆਂ ਬਿਮਾਰੀਆਂ ਲੱਗਦੀਆਂ ਜੀ? ਬਿਮਾਰੀਆਂ ਆਈ ਆ ਮਤਲਬ ਤੇਜੀਆਂ ਵਗੈਰਾ, ਹਾਂਜੀ। ਮਾੜੀ ਮੋਟੀ ਕੁੰਗੀ ਕੰਗੀ ਜੀ, ਇੱਕ ਨਾ ਚਿਰ ਥੋੜੀ ਮੋਟੀ ਆ ਜਿਹੜੀ ਹੁੰਦੀ ਹੀ ਰਹਿੰਦੀ ਐ। ਹਾਂਜੀ। ਹਾਂ। ਠੀਕ ਐ ਜੀ। ਤੇ ਉਸ ਤੋਂ ਬਾਅਦ ਸਰ ਆਪਣਾ ਧਾਨਾ ਵਿੱਚ? ਧਾਨਾ ਵਿੱਚ ਵੀ ਆ ਥੋੜ੍ਹਾ ਜਿਹਾ ਪੱਤਾ ਮਰੋੜ ਹੋ ਜਾਂਦਾ। ਹਾਂਜੀ। ਵਿੱਚ ਬੂਟੇ ਗੱਠੇ ਸੁੱਕਣ ਲੱਗ ਜਾਂਦੇ ਐ। ਠੀਕ ਐ ਜੀ। ਅਸੀਂ ਇਹੀ ਮਾੜੀ-ਮੋਟੀ ਬਿਮਾਰੀਆਂ ਹੁੰਦੀਆਂ ਹੀ ਆ। ਠੀਕ ਐ ਜੀ। ਕੀੜ

## Save Gurmukhi Transcript

In [16]:
import os
from google.colab import drive

# 1. Ensure Drive connection remains active
try:
    drive.mount('/content/drive')
except Exception:
    pass

# 2. Define the final base database directory path
base_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Using Gemini Models"

# 3. Check for the valid key string generated in Cell 2
if 'video_id' not in locals() and 'video_id' not in globals():
    raise NameError("Variable 'video_id' not found. Please make sure to run your file selection cell first!")

# 4. Dynamically target the file name subfolder
target_folder = os.path.join(base_path, video_id)
os.makedirs(target_folder, exist_ok=True)

# 5. UPDATED: Lowercases the file name identifier and appends _gurumukhi.txt
file_save_name = f"{video_id.lower()}_gurumukhi.txt"
transcript_path = os.path.join(target_folder, file_save_name)

# 6. Save transcript to the destination file
with open(transcript_path, "w", encoding="utf-8") as f:
    f.write(transcription_response.text)

print(f"✨ Success! File saved to: {transcript_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✨ Success! File saved to: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Using Gemini Models/MarauliKhurad3/maraulikhurad3_gurumukhi.txt


## Gurmukhi to Devanagari Conversion and Save

In [17]:
import os
from google.colab import drive

# 1. Ensure Drive connection remains active
try:
    drive.mount('/content/drive')
except Exception:
    pass

# 2. Define the final base database directory path
base_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Using Gemini Models"

# 3. Guard rails to ensure previous cell data is present in memory
if 'video_id' not in locals() and 'video_id' not in globals():
    raise NameError("Variable 'video_id' not found. Please run the file selection cell first!")
if 'transcription_response' not in locals() and 'transcription_response' not in globals():
    raise NameError("Variable 'transcription_response' not found. Please run the transcription cell first!")

print("Converting Gurmukhi transcript to Devanagari script...")

# 4. Use Gemini to precisely convert the Gurmukhi text into Devanagari script characters
conversion_prompt = f"""
You are a high-precision linguistic conversion model. Transliterate the following text from Gurmukhi script (Punjabi) into Devanagari script (Hindi characters).

Strict Rules:
1. Do NOT translate the language into formal textbook Hindi. Keep the exact spoken words, colloquial expressions, and dialect intact—only change the script alphabets (e.g., 'ਵੀਰ ਜੀ, ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ' must become 'वीर जी, सत श्री अकाल जी।').
2. Preserve all structural formatting, speaker tags (e.g., **ਕਿਸਾਨ:** to **किसान:**), timestamps, and bold markdown characters exactly as they appear.

Gurmukhi Source Text:
{transcription_response.text}
"""

devanagari_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=conversion_prompt
)

# 5. Target the folder path and assign the corrected file extension name
target_folder = os.path.join(base_path, video_id)
os.makedirs(target_folder, exist_ok=True)

file_save_name = f"{video_id.lower()}_devanagari.txt"
transcript_path = os.path.join(target_folder, file_save_name)

# 6. Save the true Devanagari converted text to your Google Drive database folder
# Check if devanagari_response.text is not None before writing
if devanagari_response.text:
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(devanagari_response.text)
    print(f"✨ Success! Script converted and file saved to: {transcript_path}")
else:
    print(f"❌ Warning: No Devanagari text was generated for {video_id}. File not saved.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Converting Gurmukhi transcript to Devanagari script...
✨ Success! Script converted and file saved to: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Using Gemini Models/MarauliKhurad3/maraulikhurad3_devanagari.txt
